# Evaluation — RailGuard AI violation detection

## Methodology

**Precision (เชิงปริมาณ):** ทุก event ที่ระบบตรวจจับได้ (32 เหตุการณ์จาก 9 segments, 4 จุดทางตัด)
ถูกรีวิวโดยมนุษย์จาก snapshot + คลิปหลักฐาน ให้ verdict `TP`/`FP`
ตามนิยาม: *ผู้ใช้ทางข้ามเส้นหยุดขณะรถไฟอยู่ในเขตทางตัด หรือภายในช่วง clearance (~6 วิ) หลังขบวนพ้น*
โดยกรณีกำกวม (เช่น เจ้าหน้าที่เดินในเขต) ตัดเป็น FP แบบ conservative
→ verdicts อยู่ที่ `backend/tests/fixtures/event_verdicts.csv`

**Recall (เชิงคุณภาพ):** footage ต้นฉบับเป็นกล้องมือถือ ผู้ฝ่าฝืนข้ามเส้นในเวลา <2 วินาที
การ label ground truth ครบถ้วนจาก frame strips ทำไม่ได้อย่างน่าเชื่อถือ จึงรายงานเป็นการวิเคราะห์กรณีพลาด
(ดูหัวข้อ Known misses ด้านล่าง) — เป็นข้อจำกัดที่ประกาศตรงไปตรงมา ไม่ปั้นตัวเลข

In [ ]:
import subprocess
out = subprocess.run(["../backend/.venv39/bin/python", "../backend/scripts/eval_precision.py"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)

## ผลลัพธ์ (สรุป)

| segment | TP | FP | precision |
|---|---|---|---|
| asok_a | 1 | 0 | 1.00 |
| asok_b | 1 | 1 | 0.50 |
| donmueang_a | 2 | 0 | 1.00 |
| kmitl_a | 13 | 1 | 0.93 |
| kmitl_b | 1 | 4 | 0.20 |
| kmitl_c | 2 | 0 | 1.00 |
| ram_b | 2 | 0 | 1.00 |
| ram_c | 4 | 0 | 1.00 |
| **รวม** | **26** | **6** | **0.81** |

*(หมายเหตุ: รอบแรก danger zone วาดหยาบ จับได้ 43 เหตุการณ์ (precision 0.86)
หลังปรับ zone ให้แนบแนวรางจริงตามภูมิศาสตร์ เหตุการณ์เหลือ 32 —
zone ที่ตรงความจริงสำคัญกว่าตัวเลขจำนวน event)*

## บทวิเคราะห์ False Positives (6 กรณี)
1. **ตู้รถไฟถูก track เป็น bus แล้วชนเส้นหยุดเอง** (asok_b) → แก้ได้โดย exclude track ที่กว้างผิดปกติ/ทับ zone มาก
2. **เจ้าหน้าที่จราจรเดินในเขตปิด** (kmitl_a, kmitl_b ×2) → ต้องการ uniform classifier หรือ whitelist โซนยืนเจ้าหน้าที่
3. **รถหัวแถวคืบข้ามเส้น (queue creep)** (kmitl_b) → เพิ่ม minimum displacement ก่อนนับเป็นการข้าม

## Known misses (recall qualitative)
- ช่วง surge หลังรถไฟผ่านที่ ram_c มีผู้ฝ่าฝืนหลายรายแต่ระบบจับได้บางส่วน —
  track id churn ในฝูงมอเตอร์ไซค์หนาแน่น + ผู้ข้ามนอกแนวเส้นหยุดที่วาดไว้
- ram_a (มุมสูงกว้างมาก) จับไม่ได้เลย: วัตถุเล็กเกินไปสำหรับ YOLO11s ที่ 720p → ต้องใช้ crop/tiling หรือโมเดลใหญ่ขึ้น
- การ tighten zone ให้ตรงรางแลกกับ recall ที่ลดลง (ram_b 7→2, ram_c 10→4):
  ผู้ข้ามบางแนวไม่ตัดผ่าน stop line เส้นเดียวที่นิยามไว้ (อนาคต: หลาย stop line ต่อ site)